# 13. 现代开源模型家族与新架构体系

现代大模型已经不再是“只有 Transformer 一条路”。

当前值得重点理解的路线至少包括：

- **Dense Transformer 家族**：Llama、Qwen、Gemma、OLMo
- **Sparse MoE 家族**：Mixtral、DeepSeek-V3、Qwen3 MoE
- **Hybrid SSM-Transformer 家族**：Jamba
- **State Space Models / Mamba 家族**：Mamba、Mamba-2

这一节的重点是建立“模型家族视角”，理解它们为什么这样设计。

## 现代开源模型家族与新架构的形式化定义

“模型家族”不是单一 checkpoint，而是一组共享设计哲学、训练路线和部署目标的模型集合。现代开源模型家族的差异主要不再体现在“是否使用 Transformer”，而更多体现在：结构细节、稀疏策略、长上下文能力、生态定位与开放程度。

## 输入、输出与参数化方式

不同家族的输入输出接口往往相似，通常都服务于 token 级自回归预测；真正不同的是其内部状态更新机制和系统目标。例如 Dense Transformer 家族更强调通用性，MoE 家族更强调容量扩展，SSM 家族更强调长序列效率。

## 结构分解与信息流

            本节从架构路线而非 benchmark 名次出发，将代表性家族分为：

- Dense Transformer 家族
- Sparse MoE Transformer 家族
- Hybrid SSM-Transformer 家族
- 纯 SSM / Mamba 家族

这种划分更有助于理解模型演化逻辑，因为它直接对应了不同架构对“容量、长序列、效率、开放性”的不同回答方式。

## 优化目标与训练机制

            不同家族虽然共享语言建模目标，但训练和设计目标并不相同：

- 有的优先追求通用能力与生态覆盖
- 有的优先追求单位推理成本下的容量
- 有的优先追求长上下文和吞吐
- 有的优先追求 fully-open 研究复现

因此，“为何采用该架构”必须和“该家族想优化什么目标”一起理解。

## 计算复杂度、统计性质与工程代价

            Dense、MoE、Hybrid、SSM 在复杂度、实现成熟度和系统生态上各有代价。

- Dense：成熟、稳定，但参数和推理成本耦合更强
- MoE：容量大，但路由和通信复杂
- Hybrid：长序列效率更好，但工程链路更复杂
- SSM：理论效率优越，但生态尚在发展

## 与相邻模型的差异

            家族对比的正确方法不是只比较某个 benchmark 数字，而是比较：

- 核心架构假设
- 扩展路径
- 系统代价
- 开放程度
- 适用场景

这也是为什么同样是“开源大模型”，Qwen、Gemma、OLMo、Jamba、Mamba 的学习价值并不相同。

## 先建立直觉

            学大模型不能只记模型名和参数量。

更有价值的方式是把模型看成“技术路线”：

- 有的路线优先做通用 dense 模型
- 有的路线优先做 MoE 扩容
- 有的路线试图摆脱标准注意力
- 有的路线把研究透明度放在首位

一旦你建立起“家族视角”，以后看到新模型就不会只剩下“又多了一个名字”。

## 架构是怎么工作的

            这一节不是讲一个具体网络，而是讲一类模型的设计取向。

例如：

- Qwen3 更像完整产品矩阵
- Gemma 3 更强调轻量部署与开发者友好
- OLMo 2 更强调 fully-open 科研复现
- Jamba 试图把 SSM 和 Transformer 融合
- Mamba 代表非注意力路线的重要探索

所以这节最重要的问题不是“谁更强”，而是“为什么它们走了不同路”。

## 训练时到底发生了什么

            不同架构路线会把训练难点放在不同地方：

- Dense Transformer：规模化训练、数据与推理优化
- MoE：路由、通信、专家均衡
- SSM：算子实现、序列建模能力和生态适配
- Hybrid：如何同时兼顾两类结构的优势

学会从训练难点倒推架构目标，是理解模型演化非常重要的方法。

## 什么时候该用它

            这节最适合在你已经学过 Transformer 和 MoE 之后再读。

读的时候可以重点问自己：

- 这个家族要解决什么现实问题？
- 它在哪个维度做了取舍？
- 它更适合科研、产品、长上下文还是轻量部署？

## 最常见的误区

            常见误区：

1. **误以为参数量就是全部**
   很多架构差异体现在训练配方、推理优化和系统设计上。

2. **误以为新架构一出现就会替代 Transformer**
   现实里新架构通常需要很长时间才能建立完整生态。

3. **误以为模型家族对比就是 benchmark 对比**
   benchmark 只能说明某个切面，不能代替架构理解。

## 1. 从架构目标看不同家族

### Dense Transformer

目标：在通用语言建模能力上取得稳定上限，生态最成熟。

### MoE Transformer

目标：在不让单次推理成本线性爆炸的情况下扩大总参数容量。

### SSM / Mamba

目标：缓解注意力的二次复杂度问题，用线性时间建模长序列。

### Hybrid SSM-Transformer

目标：同时保留 Transformer 的高质量建模能力与 SSM 的长序列效率。

## 2. Mamba / SSM 的核心递推公式

经典状态空间模型可写成：

$$
h_t = A h_{t-1} + B x_t
$$

$$
y_t = C h_t + D x_t
$$

Mamba 的关键创新之一在于：让参数对输入具有选择性（selective），从而增强表达能力。

相比标准注意力，SSM 的一个核心卖点是：

- 计算和显存更接近线性随序列长度增长
- 更适合超长序列或吞吐优先场景

但它也面临：

- 与注意力机制不同的归纳偏置
- 训练和生态成熟度不如 Transformer 家族

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
# 对比不同架构在理论复杂度上的增长趋势。
seq_lengths = np.array([1_000, 2_000, 4_000, 8_000, 16_000, 32_000])
attn_complexity = seq_lengths ** 2
linear_complexity = seq_lengths

complexity_df = pd.DataFrame(
    {
        "序列长度": seq_lengths,
        "Attention 复杂度(归一化)": attn_complexity / attn_complexity[0],
        "Linear/SSM 复杂度(归一化)": linear_complexity / linear_complexity[0],
    }
)
complexity_df

In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(complexity_df["序列长度"], complexity_df["Attention 复杂度(归一化)"], marker="o", label="Attention O(n^2)")
plt.plot(complexity_df["序列长度"], complexity_df["Linear/SSM 复杂度(归一化)"], marker="o", label="SSM / Linear O(n)")
plt.xscale("log")
plt.yscale("log")
plt.title("不同序列建模架构的复杂度增长趋势")
plt.xlabel("sequence length")
plt.ylabel("normalized complexity")
plt.legend()
plt.show()

## 3. 代表性模型家族对比

下面这张表整理了几类非常有代表性的现代开源 / 开放权重模型。

我这里特别用了**绝对日期**来避免“最新”理解偏差：

- `2024-11-26`：OLMo 2
- `2025-03-12`：Gemma 3
- `2025-04-29`：Qwen3
- `2024-08-22`：Jamba 1.5
- `2024-05-31`：Mamba-2 论文公开（ICML 2024 版本）

In [ ]:
family_df = pd.DataFrame(
    [
        {
            "模型家族": "Qwen3",
            "日期": "2025-04-29",
            "代表模型": "Qwen3-235B-A22B / Qwen3-32B",
            "核心架构": "Dense + MoE 并存的 Decoder-only Transformer",
            "关键机制": "Hybrid thinking, GQA, Dense+MoE lineup",
            "上下文": "32K-128K",
            "开源特征": "多尺寸覆盖极广，推理与训练路线都完整",
            "类别": "Transformer",
        },
        {
            "模型家族": "Gemma 3",
            "日期": "2025-03-12",
            "代表模型": "Gemma 3 27B",
            "核心架构": "轻量 Dense Transformer / 多模态版本",
            "关键机制": "128K context, vision-language, quantized variants",
            "上下文": "128K",
            "开源特征": "单卡友好，强调开发者落地与移动端生态",
            "类别": "Transformer",
        },
        {
            "模型家族": "OLMo 2",
            "日期": "2024-11-26",
            "代表模型": "OLMo-2-13B",
            "核心架构": "Fully Open Dense Transformer",
            "关键机制": "开放数据、代码、recipe、checkpoint",
            "上下文": "官方重点不在超长上下文，而在 fully-open 研究",
            "开源特征": "最强调 fully-open science",
            "类别": "Transformer",
        },
        {
            "模型家族": "Jamba 1.5",
            "日期": "2024-08-22",
            "代表模型": "Jamba 1.5 Large",
            "核心架构": "Hybrid SSM-Transformer",
            "关键机制": "长上下文、高吞吐、SSM + Transformer",
            "上下文": "长上下文能力是核心卖点",
            "开源特征": "兼顾长上下文与吞吐",
            "类别": "Hybrid",
        },
        {
            "模型家族": "Mamba / Mamba-2",
            "日期": "2023-12-05 / 2024-05-31",
            "代表模型": "Mamba-2.8B / Mamba2-2.7B",
            "核心架构": "Selective State Space Models",
            "关键机制": "线性时间序列建模、SSD duality",
            "上下文": "理论上更适合长序列伸展",
            "开源特征": "代表非注意力路线的重要探索",
            "类别": "SSM",
        },
    ]
)

family_df

In [ ]:
category_counts = family_df["类别"].value_counts().reset_index()
category_counts.columns = ["类别", "数量"]

plt.figure(figsize=(8, 5))
sns.barplot(data=category_counts, x="类别", y="数量", palette="Set2")
plt.title("课程中覆盖的现代模型架构类别")
plt.show()

In [ ]:
# 定性打分，只为帮助建立结构化直觉，不代表官方 benchmark。
radar_df = pd.DataFrame(
    [
        {"模型家族": "Qwen3", "长上下文": 4, "效率": 4, "开放程度": 4, "生态成熟度": 5, "新架构探索": 4},
        {"模型家族": "Gemma 3", "长上下文": 4, "效率": 5, "开放程度": 4, "生态成熟度": 4, "新架构探索": 3},
        {"模型家族": "OLMo 2", "长上下文": 3, "效率": 3, "开放程度": 5, "生态成熟度": 3, "新架构探索": 3},
        {"模型家族": "Jamba 1.5", "长上下文": 5, "效率": 5, "开放程度": 4, "生态成熟度": 2, "新架构探索": 5},
        {"模型家族": "Mamba-2", "长上下文": 5, "效率": 5, "开放程度": 4, "生态成熟度": 2, "新架构探索": 5},
    ]
)

score_long = radar_df.melt(id_vars="模型家族", var_name="维度", value_name="分数")
plt.figure(figsize=(12, 6))
sns.barplot(data=score_long, x="维度", y="分数", hue="模型家族")
plt.title("不同模型家族的定性对比")
plt.ylim(0, 5.5)
plt.show()

## 4. 如何理解这些家族的技术取向

### Qwen3

更像一个“完整产品矩阵”：

- 有 dense
- 有 MoE
- 有 reasoning / non-thinking 控制
- 有 agentic coder 路线

### Gemma 3

更强调：

- 单卡 / 单加速器友好
- 轻量、高效
- 方便开发者实际部署

### OLMo 2

最值得学习的不是单点 benchmark，而是：

- 真正公开 recipe
- 真正可复现实验路径
- 更适合科研与教学

### Jamba / Mamba

最值得学习的是它们在问一个更基础的问题：

“长序列建模是否必须依赖标准注意力？”

## 5. 官方资料链接

- Qwen3 官方博客（2025-04-29）: https://qwenlm.github.io/blog/qwen3/
- Gemma 3 官方博客（2025-03-12）: https://blog.google/technology/developers/gemma-3/
- OLMo 2 官方博客（2024-11-26）: https://allenai.org/blog/olmo2
- Jamba 1.5 官方博客（2024-08-22）: https://www.ai21.com/blog/announcing-jamba-model-family/
- Jamba 官方博客（2024-03-28）: https://www.ai21.com/blog/announcing-jamba
- Mamba 官方仓库: https://github.com/state-spaces/mamba
- Mamba 论文: https://arxiv.org/abs/2312.00752
- Mamba-2 / SSD 论文: https://arxiv.org/abs/2405.21060